# Step 4: Configure Secrets & Deploy Workflow

![GitHub Actions](https://img.shields.io/badge/GitHub_Actions-2088FF?logo=githubactions&logoColor=white)
![GitHub CLI](https://img.shields.io/badge/GitHub_CLI-181717?logo=github&logoColor=white)

This notebook sets up GitHub organization secrets and optionally deploys the workflow template to your repositories.

**What this notebook does:**
1. Loads configuration from `config.json`
2. Sets organization-level secrets (tenant ID, subscription ID, client IDs)
3. Optionally pushes the workflow template to target repositories

> **Prerequisite:** Run notebooks 01–03 first, or have your identities and environments already set up.

## Load Configuration

In [ ]:
import subprocess, json, sys, os

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Load config — search multiple likely locations
config = {}
config_path = None

# Get the project root (parent of notebooks/)
_project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in dir() else None
search_paths = [
    os.path.join(os.getcwd(), "config.json"),                        # kernel cwd
    os.path.join(os.getcwd(), "notebooks", "config.json"),           # cwd/notebooks
    os.path.join(os.getcwd(), "..", "config.json"),                  # parent of cwd
]
if _project_root:
    search_paths.append(os.path.join(_project_root, "config.json"))
    search_paths.append(os.path.join(_project_root, "notebooks", "config.json"))

# Also try a known absolute path as last resort
search_paths.append("/home/pradk/projects/federated-credentials-automation/notebooks/config.json")
search_paths.append("/home/pradk/projects/federated-credentials-automation/config.json")

for p in search_paths:
    p = os.path.normpath(p)
    if os.path.exists(p):
        config_path = p
        break

if config_path:
    with open(config_path) as f:
        config = json.load(f)
    print(f"✓ Loaded config from {config_path}")
else:
    print(f"⚠ config.json not found. Searched:")
    seen = set()
    for p in search_paths:
        n = os.path.normpath(p)
        if n not in seen:
            seen.add(n)
            print(f"  - {n}")
    print("You'll enter values manually below.")

ORG_NAME = config.get("org_name") or input("GitHub organization name: ").strip()
ENVIRONMENTS = config.get("environments", ["dev", "staging", "production"])
IDENTITIES = config.get("identities", {})
REPOS = config.get("repos", [])

if not REPOS:
    mode = input("No repos in config. Fetch from GitHub? (y/n): ").strip().lower()
    if mode == "y":
        r = run_cmd(f"gh repo list {ORG_NAME} --limit 1000 --json name --jq '.[].name'", check=False)
        if r and r.returncode == 0:
            REPOS = [line.strip() for line in r.stdout.strip().split("\n") if line.strip()]
        else:
            print(f"✗ Failed to fetch repos: {r.stderr.strip() if r else ''}")
    else:
        repos_input = input("Enter repo names (comma-separated): ").strip()
        REPOS = [r.strip() for r in repos_input.split(",") if r.strip()]

print(f"\nOrganization: {ORG_NAME}")
print(f"Environments: {ENVIRONMENTS}")
print(f"Identities: {len(IDENTITIES)} loaded")
print(f"Repositories: {len(REPOS)}")

## Collect Secret Values

Enter the Azure credentials that GitHub Actions will use. These are set as **organization-level** secrets.

In [ ]:
# --- Collect secret values ---

TENANT_ID = config.get("tenant_id") or input("Azure Tenant ID: ").strip()
SUBSCRIPTION_ID = config.get("subscription_id") or input("Azure Subscription ID: ").strip()

# Client IDs per environment
CLIENT_IDS = {}
env_secret_map = {"dev": "AZURE_CLIENT_ID_DEV", "staging": "AZURE_CLIENT_ID_STAGING", "production": "AZURE_CLIENT_ID_PROD"}

for env in ENVIRONMENTS:
    # Try to get from config identities
    default_id = IDENTITIES.get(env, {}).get("client_id", "")
    if default_id:
        print(f"  {env}: Using client ID from config: {default_id}")
        CLIENT_IDS[env] = default_id
    else:
        CLIENT_IDS[env] = input(f"  Client ID for '{env}': ").strip()

print(f"\nTenant ID: {TENANT_ID}")
print(f"Subscription ID: {SUBSCRIPTION_ID}")
for env, cid in CLIENT_IDS.items():
    print(f"  {env} client ID: {cid}")

## Set Organization Secrets

Sets secrets at the organization level so all repositories can access them.

**Secrets to set:**
- `AZURE_TENANT_ID`
- `AZURE_SUBSCRIPTION_ID`
- `AZURE_CLIENT_ID_DEV`
- `AZURE_CLIENT_ID_STAGING`
- `AZURE_CLIENT_ID_PROD`

In [ ]:
# --- Set Organization Secrets ---

visibility = input("Secret visibility ('all', 'private', or 'selected'): ").strip().lower()
if visibility not in ("all", "private", "selected"):
    visibility = "private"
    print(f"  Defaulting to 'private'")

secrets = {
    "AZURE_TENANT_ID": TENANT_ID,
    "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID,
}
for env in ENVIRONMENTS:
    secret_name = env_secret_map.get(env, f"AZURE_CLIENT_ID_{env.upper()}")
    secrets[secret_name] = CLIENT_IDS[env]

for name, value in secrets.items():
    print(f"Setting {name}...")
    cmd = f'echo "{value}" | gh secret set "{name}" --org "{ORG_NAME}" --visibility "{visibility}"'
    r = run_cmd(cmd, check=False)
    if r and r.returncode == 0:
        print(f"  ✓ Set: {name}")
    else:
        print(f"  ✗ Failed: {name}")
        print(f"    {r.stderr.strip()[:200] if r else ''}")

print("\n✓ Organization secrets configured.")

## Deploy Workflow Template (Optional)

Optionally push the `workflow_template.yml` to your repositories as `.github/workflows/deploy.yml`.

In [ ]:
# --- Deploy Workflow Template ---

deploy = input("Deploy workflow template to repos? (y/n): ").strip().lower()

if deploy == "y":
    # Read the workflow template
    template_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "workflow_template.yml")
    if not os.path.exists(template_path):
        print(f"✗ Workflow template not found at {template_path}")
    else:
        with open(template_path) as f:
            workflow_content = f.read()

        import base64
        encoded = base64.b64encode(workflow_content.encode()).decode()

        for repo in REPOS:
            print(f"\nDeploying to {repo}...")
            # Check if file already exists
            check_cmd = (
                f'gh api '
                f'"/repos/{ORG_NAME}/{repo}/contents/.github/workflows/deploy.yml" '
                f'--jq .sha 2>/dev/null'
            )
            r_check = run_cmd(check_cmd, check=False)
            sha_param = ""
            if r_check and r_check.returncode == 0 and r_check.stdout.strip():
                sha = r_check.stdout.strip()
                sha_param = f'-f "sha={sha}"'
                print(f"  Updating existing workflow (sha: {sha[:8]}...)")

            cmd = (
                f'gh api --method PUT '
                f'"/repos/{ORG_NAME}/{repo}/contents/.github/workflows/deploy.yml" '
                f'-f "message=Add Azure OIDC deployment workflow" '
                f'-f "content={encoded}" '
                f'{sha_param}'
            )
            r = run_cmd(cmd, check=False)
            if r and r.returncode == 0:
                print(f"  ✓ Deployed workflow to {repo}")
            else:
                output = (r.stdout + r.stderr) if r else ""
                print(f"  ✗ Failed: {output.strip()[:200]}")

        print("\n✓ Workflow deployment complete.")
else:
    print("Skipped workflow deployment.")
    print("You can manually copy workflow_template.yml to each repo's .github/workflows/ directory.")

## Next Steps

Proceed to **[05_verify_and_troubleshoot.ipynb](05_verify_and_troubleshoot.ipynb)** to verify the setup and troubleshoot any issues.

Or, test a deployment manually:
1. Go to any repo → **Actions** tab → **Deploy to Azure** workflow
2. Click **Run workflow** → select an environment → click **Run workflow**
3. Watch the run — it should authenticate via OIDC and show `az account show` output